T4 settings vs the local 4GB defaults: **batch_size 32, grad_accum_steps 2** — same effective batch of 64, roughly half the activation memory. The T4 reports ~14.6GiB *usable* VRAM, so batch 64 in one pass OOMs (the VRAM guard will warn if you try). Model architecture and all seeds/schedules are unchanged.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Confirm the GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 2. Get the code + dependencies

Colab already has a CUDA build of torch — we only install the missing packages (do **not** reinstall torch here; `requirements.txt`'s cu130 pin is for the local Windows machine).


In [ ]:
!git clone https://github.com/Bit-Sahil04/toy-pixel-diffuser.git

%cd toy-pixel-diffuser

!pip install -q datasets huggingface_hub pillow numpy tqdm matplotlib


In [ ]:
# 3. Environment sanity check (CUDA + AMP on the T4)

!python check_env.py


## 4. (Recommended) Mount Google Drive for checkpoints

Colab disks are wiped when the runtime dies. This saves checkpoints/sample grids/logs to Drive so a disconnected session loses at most one checkpoint interval.

Skip this cell to train fully ephemeral.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/pixel_diffuser/run1'

DRIVE_SAMPLES = '/content/drive/MyDrive/pixel_diffuser/run1_samples'

DRIVE_LOG = '/content/drive/MyDrive/pixel_diffuser/run1_log.csv'


## 5. Download + inspect the dataset

~300MB (LPC 4-view sprites, 50k images @ 128x128 RGBA). Prints real dims/modes and a sample caption; asserts no flip/rotation transforms.


In [ ]:
!python data.py


## 6. (Optional) Quick smoke test first

60 steps on 512 images to confirm the T4 setup end-to-end before the real run (~2 min, most of it the fixed-seed sample grid).


In [ ]:
# 6. (Optional) Quick smoke test first
# 60 steps on 512 images to confirm the T4 setup end-to-end before the real
# run (~3 min, most of it the fixed-seed sample grid).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py --limit 512 --max_train_steps 60 --batch_size 32 --grad_accum_steps 2


## 7. Pre-flight checklist - REQUIRED before training

Automated version of `docs/preflight_checklist.md`. Run the four cells below in order and
fix any FAIL before continuing. ~10-15 min total (most of it is the tiny-overfit sample pass
and the resume-test grid).

| cell | check | pass condition |
|---|---|---|
| 1a | tiny-overfit on 16 REAL sprites | samples reproduce targets, mean NCC > 0.6 |
| 1b | scheduler math + **visual** noise strip | closed form + diffusers AGREE; image dissolves at the right rate |
| 1c | config-effect summary | attention/prediction/EMA/batch actually as intended |
| 1d | throughput, ETA, resume integrity | GPU util high, no loss spike after resume |

**Why:** loss going down proves the training loop works - and proves nothing about the
sampler, the config, or anything downstream. We once had a healthy loss curve while the
sampler's posterior math was wrong; every grid it produced was degraded. These cells give
you a second, independent, cheap signal before you commit GPU-hours.

In [ ]:
# 1a - Tiny-overfit test (checklist #1; highest signal per minute spent)
# 16 REAL sprites -> 300 overfit steps -> full 1000-step DDPM sample (raw weights:
# EMA warmup would lag an overfit this short). PASS: outputs near-perfectly reproduce
# the inputs (mean NCC > 0.6; a real overfit should reach >0.9). FAIL: static/blank/
# unstructured output at converged loss = structural bug (sampler/model/data). STOP.
import torch
import matplotlib.pyplot as plt
from config import Config
from data import PixelArtDataset, ensure_data, denormalize
from diffusion import Diffusion
from model import build_model
from train import set_seed

OVERFIT_STEPS = 300
N = 16
set_seed(0)
cfg = Config(); cfg.batch_size = N; cfg.grad_accum_steps = 1
device = torch.device('cuda')
images_dir, captions_csv = ensure_data()
ds = PixelArtDataset(images_dir, captions_csv, image_size=cfg.image_size,
                     unconditional=True, limit=N)
# shuffle=False: the SAME fixed 16 sprites every run, so NCC is comparable
x0 = next(iter(torch.utils.data.DataLoader(ds, batch_size=N)))['image']
x0 = x0.to(device).to(memory_format=torch.channels_last)

model = build_model(cfg).to(device).to(memory_format=torch.channels_last)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
scaler = torch.amp.GradScaler('cuda')
diff = Diffusion(cfg.timesteps, cfg.beta_schedule, cfg.prediction_target, device)

losses = []
model.train()
for step in range(1, OVERFIT_STEPS + 1):
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda'):
        loss = diff.training_losses(model, x0)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    losses.append(loss.item())
    if step == 1 or step % 50 == 0:
        print(f'overfit step {step:4d} | loss {loss.item():.4f}')

assert losses[-1] < losses[0] * 0.2, 'loss did not fall - fix training path before sampling'

samples = diff.sample(model, N, cfg.image_size, seed=0)   # full 1000-step sampler, [0,1]

def ncc(a, b):  # normalized cross-correlation, per image
    a = a.flatten(1); a = a - a.mean(1, keepdim=True)
    b = b.flatten(1); b = b - b.mean(1, keepdim=True)
    return ((a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)).cpu()

scores = ncc(samples, denormalize(x0).cpu())
mean_ncc = scores.mean().item()
print('per-image NCC:', ' '.join(f'{s:.2f}' for s in scores))
verdict = 'PASS' if mean_ncc > 0.6 else 'FAIL - do NOT start a real run'
print(f'mean NCC = {mean_ncc:.3f} -> {verdict}')
assert mean_ncc > 0.6, 'tiny-overfit FAILED: structural bug - see docs/preflight_checklist.md #1'

fig, axes = plt.subplots(2, N, figsize=(N * 1.35, 3.1))
orig = denormalize(x0).cpu()
for i in range(N):
    axes[0, i].imshow(orig[i].permute(1, 2, 0).clamp(0, 1)); axes[0, i].axis('off')
    axes[1, i].imshow(samples[i].permute(1, 2, 0).clamp(0, 1)); axes[1, i].axis('off')
plt.suptitle(f'tiny-overfit: targets (top) vs 1000-step DDPM samples (bottom) - NCC {mean_ncc:.3f}')
plt.show()

plt.figure(figsize=(5, 2.5))
plt.plot(losses); plt.title('overfit loss (should flatten)'); plt.xlabel('step'); plt.show()

In [ ]:
# 1b - Scheduler numerical + VISUAL check (checklist #2)
# Three independent confirmations of diffusion.py's schedule and posterior math:
#   (1) closed-form re-implementation computed HERE, not imported,
#   (2) cross-check against diffusers' trusted DDPMScheduler,
#   (3) a forward-noise strip so you SEE the schedule destroy a real sprite
#       at the right rate (intact to ~t=400, pure static by ~t=800).
import math
import torch
import matplotlib.pyplot as plt
from data import PixelArtDataset, ensure_data, denormalize
from diffusion import Diffusion

diff = Diffusion(1000, 'cosine', 'epsilon', 'cpu')
T = diff.timesteps

# (1) independent closed form (Nichol & Dhariwal 2021 cosine).
# Apply the SAME documented clamp as diffusion.py (betas in [1e-4, 0.999]) —
# the lower clamp binds for the first ~30 steps (unclamped beta_0 = 4.1e-5)
# and shifts abar by <=0.2%; that is deliberate, not an error.
steps = torch.arange(T + 1, dtype=torch.float64)
abar_cf = torch.cos(((steps / T) + 0.008) / 1.008 * math.pi / 2) ** 2
abar_cf = abar_cf / abar_cf[0]
betas_cf = (1 - (abar_cf[1:] / abar_cf[:-1])).clamp(1e-4, 0.999)
abar_cf = torch.cumprod(1 - betas_cf, dim=0)
ok = torch.allclose(diff.alphas_cumprod, abar_cf.float(), atol=1e-5)
print(f"(1) abar vs closed form (all 1000 t, same clamp): "
      f"{'AGREE' if ok else 'DISAGREE - DO NOT TRAIN'}")
assert ok

# (2) trusted-reference diff
try:
    from diffusers import DDPMScheduler
    ref = DDPMScheduler(num_train_timesteps=1000, beta_schedule='cosine')
    dmax = (ref.betas - diff.betas).abs().max().item()
    agree = torch.allclose(ref.betas, diff.betas, atol=1e-4)
    print(f"(2) betas vs diffusers DDPMScheduler(cosine): "
          f"{'AGREE' if agree else 'DISAGREE'} (max diff {dmax:.2e})")
except ImportError:
    print('(2) SKIP: diffusers not installed - %pip install diffusers, then re-run')
else:
    # diffusers does NOT lower-clamp, so beta_0 differs slightly (6e-5);
    # atol=1e-4 tolerates exactly that documented difference
    print('    (diffusers applies no 1e-4 lower clamp; ours does, by design — '
          'diff <= 6e-5 at t~0 only)')

# (3) printed spot-table: the exact values the sampler will use
print(f"{'t':>5} {'beta_t':>9} {'abar_t':>9} {'abar_prev':>10} "
      f"{'coef_x0':>9} {'coef_xt':>9} {'post_var':>9}")
for i in (1, 100, 500, 999):
    abar, abar_p = diff.alphas_cumprod[i], diff.alphas_cumprod[i - 1]
    bt, at = diff.betas[i], 1.0 - diff.betas[i]
    den = 1.0 - abar
    cx0 = (abar_p.sqrt() * bt / den).item()
    cxt = (at.sqrt() * (1.0 - abar_p) / den).item()
    var = (bt * (1.0 - abar_p) / den).item()
    print(f'{i:5d} {bt.item():9.5f} {abar.item():9.5f} {abar_p.item():10.5f} '
          f'{cx0:9.5f} {cxt:9.5f} {var:9.6f}')

# (4) VISUAL: schedule curves + forward-noise strip on a REAL sprite
abar_prev = torch.cat([torch.ones(1), diff.alphas_cumprod[:-1]])
post_var = diff.betas * (1 - abar_prev) / (1 - diff.alphas_cumprod)
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
ax[0].plot(diff.betas); ax[0].set_title('beta_t (per-step noise)')
ax[1].plot(diff.sqrt_alphas_cumprod, label='sqrt(abar): signal kept')
ax[1].plot(diff.sqrt_one_minus_alphas_cumprod, label='sqrt(1-abar): noise added')
ax[1].legend(); ax[1].set_title('signal vs noise over t')
ax[2].plot(post_var); ax[2].set_title('posterior variance')
plt.tight_layout(); plt.show()

ds = PixelArtDataset(*ensure_data(), image_size=128, unconditional=True, limit=8)
img = ds[0]['image'][None]                    # one REAL sprite in [-1,1]
ts = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
g = torch.Generator().manual_seed(3)
fig, axes = plt.subplots(1, len(ts), figsize=(len(ts) * 1.3, 1.8))
for a, t in zip(axes, ts):
    xt = diff.q_sample(img, torch.tensor([t]), torch.randn(img.shape, generator=g))
    a.imshow(denormalize(xt)[0].permute(1, 2, 0).clamp(0, 1))
    a.set_title(f't={t}', fontsize=8); a.axis('off')
plt.suptitle('q_sample on ONE real sprite: expect intact to ~t=400, ghost ~t=600, '
             'pure static by ~t=800')
plt.show()

In [ ]:
# 1c - Config sanity: what the config ACTUALLY does (checklist #3)
# Classic silent bug this catches: attention_resolutions=(16, 8) never matching
# real feature maps of [128, 64, 32] under exact-match semantics.
import torch
from config import Config
from model import SelfAttention, build_model
from train import print_config_summary

cfg = Config()
model = build_model(cfg)
print_config_summary(cfg, model)   # same summary train.py prints at startup

feat_sizes = [cfg.image_size // 2 ** i for i in range(len(cfg.channel_mults))]
in_cfg = [s for s in feat_sizes if s in cfg.attention_resolutions]
le_cfg = [s for s in feat_sizes if s <= max(cfg.attention_resolutions)]
print(f'feature sizes: {feat_sizes} | attention_resolutions={cfg.attention_resolutions}')
print(f'  exact-match semantics (`in`): would fire at {in_cfg or "NOTHING"}  <- the old silent bug')
print(f'  <= semantics (actual):        fires at {le_cfg} (+ the always-on bottleneck mid)')

# runtime verification, not just prints: count what the module tree REALLY built
n_attn = sum(1 for m in model.modules() if isinstance(m, SelfAttention))
expected = 1 + 2 * len(le_cfg)   # mid + down/up at each qualifying level
assert n_attn == expected, f'attention modules {n_attn} != expected {expected}'
n_params = sum(p.numel() for p in model.parameters())
print(f'PASS: {n_attn} attention modules in built model (= {expected} expected), '
      f'{n_params / 1e6:.2f}M params')

In [ ]:
# 1d - Throughput, ETA, resume integrity (checklist #4, #5, #6, #7)
import csv
import subprocess
from pathlib import Path

print('GPU right now (idle is fine):')
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('During real training, utilization.gpu should hold ~90%+. If low: I/O or CPU bound -')
print('check per-step Drive writes, num_workers, per-step .item() syncs (checklist #5).')

from config import Config
cfg = Config()
eff = cfg.batch_size * cfg.grad_accum_steps
epochs = cfg.max_train_steps * eff / 50000
print(f'effective batch = {cfg.batch_size} x {cfg.grad_accum_steps} = {eff} | '
      f'{cfg.max_train_steps} steps ~ {epochs:.0f} epochs over 50k images')

# throughput from the smoke run's CSV (cell 6 wrote logs/train_log.csv)
rows = list(csv.reader(open('logs/train_log.csv')))
steps_done = len(rows) - 1
if steps_done >= 10:
    sps = float(rows[-1][3]) / steps_done
    train_h = sps * cfg.max_train_steps / 3600
    grids = cfg.max_train_steps // cfg.sample_every + 1
    grid_h = grids * 12 / 60                      # ~12 min per 4x4 grid on T4
    print(f'measured {sps:.2f} s/step (smoke run) -> training ~{train_h:.1f} h')
    print(f'grids: {grids} x ~12 min at --sample_every {cfg.sample_every} = ~{grid_h:.1f} h')
    print(f'BUDGET ~{train_h + grid_h:.1f} h total. Decide now what 25/50/100% progress '
          f'looks like (docs/preflight_checklist.md #7).')
else:
    print('run the smoke-test cell first for a throughput estimate')

# resume integrity on an EPHEMERAL dir (never touches your Drive checkpoints).
# NOTE: the resume run emits one sample grid at its first step by design - that is
# checklist #4\'s "look at an undertrained grid" (expect blurry blobs / flat colors,
# NOT static noise). ~6 min on T4.
env = 'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True '
tmp = Path('/content/preflight_resume'); (tmp / 's').mkdir(parents=True, exist_ok=True)
common = (f'--limit 512 --batch_size 32 --grad_accum_steps 2 --sample_every 100000 '
          f'--checkpoint_dir {tmp} --samples_dir {tmp}/s --log_csv {tmp}/log.csv')

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1000:])
    assert r.returncode == 0, r.stderr[-1500:]
    return r.stdout

run(env + f'python train.py --max_train_steps 40 --ckpt_every 40 {common}')
run(env + f'python train.py --max_train_steps 42 --resume_from {tmp}/latest.pt {common}')

log = list(csv.reader(open(tmp / 'log.csv')))[1:]
pre, post = float(log[-3][1]), float(log[-2][1])
print(f'last pre-resume loss {pre:.4f} -> first post-resume loss {post:.4f}')
if post < pre * 3:
    print('PASS: resume continues smoothly (optimizer + AMP + EMA state restored)')
else:
    print('WARN: loss spiked after resume - optimizer state likely not restored; investigate')

## 8. Train

Defaults: 20k steps, effective batch 64 (32x2 accum), checkpoint every 500 steps, fixed-seed 4x4 grid every **1000** steps (override with `--sample_every`). Grids cost ~12 min each on T4, so keep the cadence coarse - the grids, not the training, dominate wall-clock.

**Interrupted?** Just re-run this cell with the resume line below - optimizer/EMA/AMP state restores exactly.


In [ ]:
# fresh run (total VRAM is auto-detected from the GPU).
# --sample_every 1000: each fixed-seed grid costs ~12 min on T4; the
# default 200 would spend ~1.5-2h of a 20k-step run on grids. 1000 -> ~4 grids.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py  \
    --batch_size 32 --grad_accum_steps 2 --sample_every 1000  \
    --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"

# ...or resume after an interruption:
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train.py  \
#     --batch_size 32 --grad_accum_steps 2 --sample_every 1000  \
#     --resume_from "$DRIVE_CKPT/latest.pt"  \
#     --checkpoint_dir "$DRIVE_CKPT" --samples_dir "$DRIVE_SAMPLES" --log_csv "$DRIVE_LOG"


## 9. Watch progress

Sample grids use a FIXED seed: every frame denoises the same noise, so you literally watch sprites emerge.


In [ ]:
!python make_gif.py --samples_dir "$DRIVE_SAMPLES"
from IPython.display import Image
Image(filename=f'{DRIVE_SAMPLES}/training_progress.gif')

## 10. Inference from the latest checkpoint

Uses EMA weights; 16 sprites via the full 1000-step DDPM sampler (~4 min on T4 at batch 16).


In [ ]:
!python sample.py --checkpoint "$DRIVE_CKPT/latest.pt" --n 16 --nrow 4 --save_individual
from IPython.display import Image
import glob
Image(filename=sorted(glob.glob('samples/infer_step*.png'))[-1])

## Notes
- Effective batch is 32x2 = 64. Batch 64 in a single pass does NOT fit on a T4 (heuristic estimate ~14.1GB vs ~12.4GB safe fraction of the ~14.6GiB usable) — the guard will warn; expect OOM.
- Loss CSV lives at the `$DRIVE_LOG` path; plot with `pandas` if you want curves.
- Colab free tier disconnects after idle timeouts — the resume line in cell 7 makes that a non-event.
- Same checkpoint format as the local RTX 3050 run: `latest.pt` from Colab resumes locally and vice versa.